# SDTM EX Domain Transformation

---
## Overview
Notebook 6 of 8.

This notebook transforms raw dose data into SDTM Exposure (EX) domain format.

Run the entire notebook to load the data.

---

## Source Data

- **Source:** bronze.dose_raw
- **Domain:** Exposure (EX)

---

## Transformation Steps

**1. Data Quality**  
Loads bronze data and transforms records to domain standard

**2. SDTM Mapping**  
Maps source columns to SDTM variables:
- Treatment identifiers (EXTRT, EXDOSE)
- Administration details (EXDOSFRM, EXROUTE)
- Dosing dates (EXSTDTC, EXENDTC)

**3. Standardization**  

- Repeat flag coding
- Route standardisation

---

## Target

SDTM-compliant data written to **silver.sdtm_ex** Delta table.

---

In [0]:
from pyspark.sql.functions import udf, col, lit, when
from pyspark.sql.types import StringType

In [0]:
#global variables
CATALOG = "cluepoints_db_demo"
SCHEMA = "silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}") #should exist if demog run first

DataFrame[]

In [0]:
#Decode route - assumed mapping - cannot find a SDTM variable that corresponds exactly
#See previous notebook comments about Scala UDFs

@udf(returnType=StringType())
def decode_exroute(dosinjec_code):
    if dosinjec_code == 0:
        return 'ORAL'
    elif dosinjec_code == 1:
        return 'INJECTION'
    elif dosinjec_code == 3:
        return 'INTRAVENOUS'
    return None

In [0]:
# Decode repeat flag - cannot find a SDTM variable that corresponds exactly
@udf(returnType=StringType())
def decode_repeat(repeat_code):
    if repeat_code == 0:
        return 'N'
    elif repeat_code == 1:
        return 'Y'
    return None

In [0]:
#load the dose table and drop the required rows, leave the 'probably' and 'dunno' records
dose = spark.table("cluepoints_db_demo.bronze.dose").filter(col('drop') != 'yes')

# Create spark dataframe - alias the columns and add the decode UDFs
# assuming the real SDTM domain is called 'Exposure' not 'Dose'
sdtm_ex = dose.select(
    col('subjid').alias('SUBJID'),
    col('center').cast('string').alias('SITEID'),
    col('dostrda').alias('EXSTDTC'),
    col('drugname').alias('EXTRT'),
    decode_exroute(col('dosinjec')).alias('EXROUTE'),
    col('vnum').alias('VISITNUM'),
    col('vname').alias('VISIT'),
    decode_repeat(col('repeat')).alias('EXRPT'),
    col('source_file'),
    col('loaded_at'),
    col('ingestion_id')
)


In [0]:
# Write to Delta - overwrite all target data
sdtm_ex.write.format('delta').mode('overwrite').saveAsTable('cluepoints_db_demo.silver.ex_sdtm')

In [0]:
#display the table
display(spark.sql("select * from cluepoints_db_demo.silver.ex_sdtm limit 10"))

SUBJID,SITEID,EXSTDTC,EXTRT,EXROUTE,VISITNUM,VISIT,EXRPT,source_file,loaded_at,ingestion_id
826290004,290,2004-09-30,null,ORAL,7.0,DAY7,N,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2004-11-05,null,ORAL,6.0,DAY6,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2005-11-07,null,ORAL,6.0,DAY6,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2005-03-25,null,ORAL,7.0,DAY7,N,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2005-01-29,null,ORAL,6.0,DAY6,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2005-07-08,null,ORAL,7.0,DAY7,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2004-01-04,null,ORAL,6.0,DAY6,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2005-05-29,null,ORAL,7.0,DAY7,Y,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2004-02-26,null,ORAL,6.0,DAY6,N,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
826290004,290,2004-04-02,null,ORAL,6.0,DAY6,N,dose.csv,2026-01-04T13:52:01.600Z,20260104_135159
